In [2]:
from dotenv import load_dotenv

In [3]:
load_dotenv()

True

In [7]:
from pydantic_ai import Agent

agent = Agent(
    'google:gemini-3.5-flash', 
    system_prompt='You are a helpful AI assistant who always responds with a single line of concise output.'
)
response = await agent.run('Who are you?')
print(response.output)

AgentRunResult(output='I am Gemini 3.6 Flash, a large language model built by Google.')


In [9]:
from pydantic import BaseModel

class Person(BaseModel):
    name: str
    age: int
    job: str

agent = Agent(
    'google:gemini-3.5-flash', 
    system_prompt='You are a helpful AI assistant who always responds with a structured output for people',
    output_type=Person
)
response = await agent.run('Kim is a 24 year old forward deployed engineer')
print(response.output)

name='Kim' age=24 job='forward deployed engineer'


In [19]:
agent = Agent(
    'google:gemini-3.5-flash', 
    system_prompt='You are a helpful AI assistant who always responds with a single line of concise output.',
    #tools=[get_supply_location]
)


In [20]:
@agent.tool_plain
def get_supply_location(name: str) -> str:
    """This function returns the supply location based on user"""
    if name == 'Harish':
        return "Hyderabad"
    else:
        return "Chennai"

In [22]:
response = await agent.run('What is my supply location, my name is Harish')
print(response.output)
#response = await agent.run('What is it used for', message_history=response.all_messages())
#print(response.output)
#print(response.all_messages())

Your supply location is Hyderabad.


In [26]:
import sqlite3
from dataclasses import dataclass
from pydantic_ai import RunContext

@dataclass
class Mydependencies:
    user_name: str
    db_connection: sqlite3.Connection

agent = Agent(
    'google:gemini-3.5-flash', 
    deps_type = Mydependencies,
    system_prompt='You are a helpful AI assistant who always responds by calling the user by name.',
)

@agent.system_prompt
def add_user_name(ctx: RunContext[Mydependencies]) -> str:
    return f'The user name is {ctx.deps.user_name}.'

@agent.tool
def list_all_users(ctx: RunContext[Mydependencies]) -> list[tuple]:
    """List all employess from database"""
    conn = ctx.deps.db_connection
    cursor = conn.cursor()

    cursor.execute('Select id, name from users;')
    people = cursor.fetchall()

    result = []
    for row in people:
        result.append((row[0], row[1]))

    return result

db_conn = sqlite3.connect('/Users/harishkumar/Desktop/LearnTech/Database/my_database.db', check_same_thread = False)
user_name = 'Harish'

deps = Mydependencies(user_name, db_conn)

response = await agent.run('What details about people are present?', deps=deps)
print(response.output)

Harish, the database currently contains employee IDs and names. At present, there is one user record listed: ID 1 for Harish.
